> **AI-generated documentation notice.** Every docstring and inline comment in this notebook was written by Claude Opus 4.7 (via Claude Code) from our code, then read and verified by us. The computation, methods, and results are entirely the team's own work. See `doc/FINAL/SIMC2026_AI_Usage_Log.pdf`, Entry 1.

# SIMC 2026 — Endeavour Challenge: combined solution notebook

This notebook stitches together the code for all parts of the challenge that have a
computational component, in solution order, so the whole pipeline runs top-to-bottom from
one kernel:

- **Task 1 — Markov-matrix model** (numerical validation, parts f & h). Self-contained;
  reads no external data.
- **Task 2 — Power Method & SVD** is **analytical only** (proofs of convergence, the
  spectral-gap rate, the X^T X / X X^T equivalence, and a worked 3×2 SVD by hand). It has
  **no code**; the full worked solution is in the preliminary report. A short marker cell
  sits where Task 2 would go.
- **Task 3 — Design / covariance / Gram matrices** (PCA & clustering, parts a–g). Reads
  `data/input/mystery_matrix1.npy`; writes figures to `data/output/TASK 3/`.

**How to run.** Execute cells in order. Task 3 auto-locates the repo root, so the notebook
works whether launched from `src/` or the repo root. The canonical, individually-maintained
notebooks remain `src/task1/task1.ipynb` and `src/task_3.ipynb`; this file is the merged view.


In [ ]:
# ===========================================================
# Part 1f — numerical validation of parts (d) and (e)
# ===========================================================
'''Purpose. Confirm the closed-form solution P(t) = c1*v1 + c2*(1-eps)^t * v2 (part 1d)
and its limit P(t) -> (0,1) (part 1e) by checking them against a brute-force iteration of
the recursion that uses no analytical input.

Method (two independent tracks):
  - Track A (ground truth): iterate P(t+1) = M @ P(t) for T steps.
  - Track B (closed form): decompose P(0) in the eigenbasis {v1, v2}, then evaluate the
    scalar-power formula for all t at once via broadcasting.
  The eigenpairs are taken from part 1b (lambda1=1 persistent, lambda2=1-eps decaying); the
  coefficients c1, c2 are obtained by solving the 2x2 linear system P(0) = c1*v1 + c2*v2,
  not hard-coded, so the cell also re-derives the c1=1, c2=1/2 of part 1c.

Validation & outputs. max|A - B| must be < 1e-10 (asserted, so a broken formula fails
loudly). P(T) is printed to show convergence to (0,1). Defines t, c1, c2, v1, v2, P_num,
which the 1h cells reuse — run this cell first.'''
import numpy as np

# --- Parameters ---
eps = 0.1                                 # conversion rate per generation
p0  = np.array([0.5, 0.5])                # initial state (p2 = 1 - p1)
T   = 100                                 # generations to simulate

# Markov matrix; lower-triangular, column-stochastic.
M = np.array([[1 - eps, 0.0],
              [eps,     1.0]])

# --- Stage 1: direct simulation ---
# Ground truth: iterate the recursion with no analytical input.
P_num = np.zeros((T + 1, 2))
P_num[0] = p0
for t_step in range(T):
    P_num[t_step + 1] = M @ P_num[t_step]

# --- Stage 2: closed form via eigenmode decomposition ---
# Eigenvectors from 1b (lambda_1 = 1, lambda_2 = 1 - eps).
v1 = np.array([0,  1])                    # persistent mode
v2 = np.array([1, -1])                    # decaying   mode

# Solve P(0) = c1*v1 + c2*v2 for the coefficients in the eigenbasis.
c1, c2 = np.linalg.solve(np.column_stack([v1, v2]), p0)
print(f"c1 = {c1},  c2 = {c2}")           # expect c1 = 1, c2 = 0.5

# Evaluate the closed form for all t at once via broadcasting.
t = np.arange(T + 1)
P_closed = c1 * v1 + c2 * ((1 - eps) ** t)[:, None] * v2

# --- Stage 3: validation ---
# Both arrays must agree to machine precision; the assertion catches any
# future edit that breaks the closed form.
max_err = np.max(np.abs(P_num - P_closed))
print(f"max |numeric - closed form| = {max_err:.2e}")
assert max_err < 1e-10, "closed form disagrees with simulation -- check 1d"

# Asymptotic check (validates 1e): P(T) sits near [0, 1] = nu_1.
print(f"P(T={T}) = {P_num[-1]}   (should approach [0, 1])")